# Gradio-Interface Study A
(Stimuli Study A Builder and Slot Builder, Practice-Stimuli and Slot Builder)

In [ ]:
# Study A Interface – Gradio 5.49.1 compatible
# Setup (install Gradio and Pandas)

!pip install gradio==5.49.1 pandas --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.0 MB/s eta 0:00:00


In [ ]:
# Setup: Mount drive & define paths

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import re
import time
import random
import logging
import gradio as gr
import math
import hashlib

Mounted at /content/drive


In [ ]:
# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("prep_studyA")

logger.info("=== Prepare Study A stimuli with GOLD + IMC ===")

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/StudyA_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Practice Gold-Data

In [ ]:
# once
# Load Practice Gold dialogues (for feedback during the practice phase)

PRACTICE_GOLD_GOOD_CSV = "/content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold/gold_dialogs_with_evaluations.csv"
PRACTICE_GOLD_BAD_CSV  = "/content/drive/MyDrive/Masterarbeit/Dialoge_Final_Gold/bad_gold_dialogs_with_evaluations.csv"

practice_gold_pool = []
HAS_PRACTICE_GOLD = False

try:
    pg_good = pd.read_csv(PRACTICE_GOLD_GOOD_CSV)
    pg_bad  = pd.read_csv(PRACTICE_GOLD_BAD_CSV)

    pg_all = pd.concat([pg_good, pg_bad], ignore_index=True)

    # Standardize column names as in Study-A-Stimuli
    # recipe_title & dialog_text will be used directly later
    pg_all["recipe_title"] = pg_all["recipe"]
    pg_all["dialog_text"]  = pg_all["discussion"]

    # Only relevant columns + reference rating for OVERALL
    practice_gold_pool = pg_all.to_dict(orient="records")
    HAS_PRACTICE_GOLD = len(practice_gold_pool) > 0

    logger.info(f"[PRACTICE_GOLD] loaded {len(practice_gold_pool)} gold dialogs.")
except Exception as e:
    logger.exception("[PRACTICE_GOLD] Could not load practice gold dialogs; fallback to generic practice.")
    practice_gold_pool = []
    HAS_PRACTICE_GOLD = False

In [ ]:
# save practice data as CSV (once)
if HAS_PRACTICE_GOLD:
    pg_all.to_csv('/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/gold_practice.csv', index=False)

In [ ]:
# After creating and saving the one-time practice CSV file
# Load Gold Practice dialogs

PRACTICE_GOLD_CSV = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/gold_practice.csv"

practice_gold_pool = []
HAS_PRACTICE_GOLD = False

try:
    df_practice_gold = pd.read_csv(PRACTICE_GOLD_CSV)
    # Security: log expected columns
    logger.info(f"[PRACTICE GOLD] df_practice_gold shape={df_practice_gold.shape}")
    logger.info(f"[PRACTICE GOLD] columns={list(df_practice_gold.columns)}")

    practice_gold_pool = df_practice_gold.to_dict(orient="records")
    HAS_PRACTICE_GOLD = len(practice_gold_pool) > 0
    logger.info(f"[PRACTICE GOLD] loaded {len(practice_gold_pool)} practice gold dialogs")
except Exception as e:
    logger.exception("[PRACTICE GOLD] Could not load gold_practice.csv – fallback to normal dialogs for practice.")
    practice_gold_pool = []
    HAS_PRACTICE_GOLD = False

## Load stimuli and apply Gold and IMC

In [ ]:
# Imports & Paths

GOOD_CSV = "/content/drive/MyDrive/Masterarbeit/Dialoge/dialogs_with_evaluations.csv"
BAD_CSV  = "/content/drive/MyDrive/Masterarbeit/Dialoge/bad_dialogs_with_evaluations.csv"

STIMULI_CSV = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_dialogs_from_goodbad.csv"
ASSIGN_CSV  = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_assignment_slots.csv"

# Helper: fix Dialog-Newlines (Fat/Carb)
def fix_dialog_newlines(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\*sFat:', r'\nFat:', text)
    text = re.sub(r'\*sCarb:', r'\nCarb:', text)
    return text.strip()

# Load and standardize GOOD / BAD
logger.info(f"Lade GOOD_CSV: {GOOD_CSV}")
df_good = pd.read_csv(GOOD_CSV)
logger.info(f"df_good shape: {df_good.shape}")

logger.info(f"Lade BAD_CSV: {BAD_CSV}")
df_bad = pd.read_csv(BAD_CSV)
logger.info(f"df_bad shape:  {df_bad.shape}")

# Newlines in the dialog
for df in (df_good, df_bad):
    if "discussion" in df.columns:
        df["discussion"] = df["discussion"].apply(fix_dialog_newlines)

# set condition
df_good["condition"] = "good"
df_bad["condition"]  = "bad"

# Merge and define base columns
df_all = pd.concat([df_good, df_bad], ignore_index=True).copy()
logger.info(f"df_all shape (good+bad): {df_all.shape}")
logger.info(f"df_all columns: {list(df_all.columns)}")

# Dialog text and recipe title
if "discussion" not in df_all.columns:
    raise ValueError("Column 'discussion' missing in GOOD/BAD CSVs")
if "recipe" not in df_all.columns:
    raise ValueError("Column 'recipe' missing in GOOD/BAD CSVs")

df_all["dialog_text"]  = df_all["discussion"]
df_all["recipe_title"] = df_all["recipe"]

# Unique dialog_id
df_all["dialog_id"] = [f"D{i+1:03d}" for i in range(len(df_all))]

# add recipe_type (manual mapping) inside the pipeline
if "recipe_type" not in df_all.columns:
    df_all["recipe_type"] = ""

RECIPE_TYPE_GROUPS = {
    "savory": ["D001", "D003", "D010", "D011", "D012", "D015", "D018", "D019", "D021", "D022", "D023", "D024", "D025", "D027", "D028", "D029", "D030",
               "D031", "D032", "D033", "D034", "D035", "D037", "D041", "D043", "D044", "D045", "D051", "D052", "D053", "D056", "D059", "D060", "D062",
               "D063", "D064", "D065", "D067", "D068", "D069", "D070", "D071", "D075", "D076", "D078", "D079", "D080"],
    "sweet":  ["D002", "D004", "D005", "D006", "D007", "D008", "D009", "D013", "D014", "D016", "D017", "D020", "D026", "D036", "D038", "D039", "D040",
               "D042", "D046", "D047", "D048", "D049", "D050", "D054", "D055", "D057", "D058", "D061", "D066", "D072", "D073", "D074", "D077"],
}

RECIPE_TYPE_MAP = {
    dialog_id: recipe_type
    for recipe_type, dialog_ids in RECIPE_TYPE_GROUPS.items()
    for dialog_id in dialog_ids
}

df_all["recipe_type"] = (
    df_all["dialog_id"]
    .map(RECIPE_TYPE_MAP)
    .fillna(df_all["recipe_type"])
    .fillna("")
)

missing = df_all.loc[df_all["recipe_type"].astype(str).str.strip() == "", "dialog_id"].tolist()
logger.info(f"[recipe_type] Missing recipe_type count: {len(missing)} / {len(df_all)}")
# optional print:
print("Missing recipe_type for dialog_ids:", missing[:20], "..." if len(missing) > 20 else "")
print("Missing count:", len(missing), "of", len(df_all))

# Gold / IMC basic structure

possible_overall_cols = [c for c in df_all.columns if "overall" in c.lower()]
logger.info(f"Possible overall columns: {possible_overall_cols}")

if len(possible_overall_cols) == 0:
    logger.warning("No overall column found. Gold/IMC expected values remain empty.")
    overall_col = None
else:
    overall_col = possible_overall_cols[0]
    logger.info(f"Use overall column: {overall_col}")

# Default values for gold & IMC
df_all["is_gold"]              = 0
df_all["gold_expected_min"]    = ""
df_all["gold_expected_max"]    = ""
df_all["is_imc"]               = 0
df_all["imc_expected_overall"] = ""

# Define Gold-Dialogs
GOLD_DIALOG_IDS = ["D001", "D005", "D011", "D012", "D023", "D029", "D030", "D039", # good (8)
                   "D044", "D045", "D051", "D052", "D053", "D058", "D076", "D079"] # bad (8)
logger.info(f"GOLD_DIALOG_IDS: {GOLD_DIALOG_IDS}")

if GOLD_DIALOG_IDS:
    df_all["is_gold"] = df_all["dialog_id"].isin(GOLD_DIALOG_IDS).astype(int)
    if overall_col is not None:
        overall_vals = pd.to_numeric(df_all[overall_col], errors="coerce")
        gold_min = (overall_vals.round() - 1).clip(lower=1, upper=7)
        gold_max = (overall_vals.round() + 1).clip(lower=1, upper=7)

        df_all.loc[df_all["is_gold"] == 1, "gold_expected_min"] = gold_min[df_all["is_gold"] == 1].astype(int).astype(str)
        df_all.loc[df_all["is_gold"] == 1, "gold_expected_max"] = gold_max[df_all["is_gold"] == 1].astype(int).astype(str)

# Define IMC-Dialogs
IMC_DIALOG_IDS = ["D006", "D016", "D017", "D027", "D036", # good (5)
                  "D043", "D046", "D056", "D057", "D066"] # bad (5)
logger.info(f"IMC_DIALOG_IDS: {IMC_DIALOG_IDS}")

if IMC_DIALOG_IDS and overall_col is not None:
    df_all["is_imc"] = df_all["dialog_id"].isin(IMC_DIALOG_IDS).astype(int)
    overall_vals = pd.to_numeric(df_all[overall_col], errors="coerce")
    imc_expected = overall_vals.round().astype("Int64").astype(str)
    df_all.loc[df_all["is_imc"] == 1, "imc_expected_overall"] = imc_expected[df_all["is_imc"] == 1]

# Column order for Study A
studyA_cols = [
    "dialog_id",
    "recipe_title",
    "recipe_type",
    "condition",
    "dialog_text",
    "is_gold",
    "gold_expected_min",
    "gold_expected_max",
    "is_imc",
    "imc_expected_overall",
]

if "condition" not in df_all.columns:
    df_all["condition"] = "good"

df_studyA = df_all[studyA_cols].copy()

# Save: Stimuli file for Study A
df_studyA.to_csv(STIMULI_CSV, index=False, encoding="utf-8")
logger.info(f"Study A Stimuli saved under: {STIMULI_CSV}")
display(df_studyA.head())

Missing recipe_type for dialog_ids: [] 
Missing count: 0 of 80


,dialog_id,recipe_title,recipe_type,condition,dialog_text,is_gold,gold_expected_min,gold_expected_max,is_imc,imc_expected_overall
0,D001,Moong Dal,savory,good,Fat: Moong Dal has low fat content. Carb: Most...,1,6,7,0,
1,D002,Bourbon Wieners,sweet,good,Fat: Bourbon Wieners have a notable fat conten...,0,,,0,
2,D003,Cobb Salad Ham Roll-ups,savory,good,Fat: The fat content seems relatively high due...,0,,,0,
3,D004,Grain-Free Apple Cinnamon Dutch Babies,sweet,good,Fat: This recipe has a moderate fat content fr...,0,,,0,
4,D005,Best Ever Muffins,sweet,good,Fat: The fat content seems relatively low in t...,1,5,7,0,


In [ ]:
# Create/update assignment file (slots)
if not os.path.exists(ASSIGN_CSV):
    assign_df = df_studyA[["dialog_id", "recipe_title", "recipe_type", "condition", "is_gold", "is_imc"]].copy()
    # 2 Ratings per Dialog
    assign_df["remaining_slots"] = 2
    assign_df.to_csv(ASSIGN_CSV, index=False, encoding="utf-8")
    logger.info(f"Assignment file created: {ASSIGN_CSV}")
else:
    logger.info(f"Assignment file already exists, not overwritten: {ASSIGN_CSV}")
    assign_df = pd.read_csv(ASSIGN_CSV)

    # Hard safety: dialog_id must be unique
    if "dialog_id" not in assign_df.columns:
        raise ValueError("ASSIGN_CSV is missing required column: dialog_id")

    # Drop duplicate dialog_id rows (keep first)
    before = len(assign_df)
    assign_df = assign_df.drop_duplicates(subset=["dialog_id"], keep="first").copy()
    after = len(assign_df)
    if after != before:
        logger.warning(f"[ASSIGN SAFETY] Dropped {before - after} duplicate dialog_id rows from ASSIGN_CSV")

    # Ensure remaining_slots exists
    if "remaining_slots" not in assign_df.columns:
        logger.warning("[ASSIGN MIGRATION] 'remaining_slots' missing; initializing to 2")
        assign_df["remaining_slots"] = 2

    # Columns that need to be in ASSIGN_CSV
    ensure_cols = ["recipe_title", "recipe_type", "condition", "is_gold", "is_imc"]

    # Build lookup from df_studyA (source of truth)
    src = df_studyA[["dialog_id"] + ensure_cols].copy()

    # Add missing columns
    missing_cols = [c for c in ensure_cols if c not in assign_df.columns]
    if missing_cols:
        logger.info(f"[ASSIGN MIGRATION] Adding missing cols: {missing_cols}")
        for c in missing_cols:
            assign_df[c] = pd.NA

    # Fill any missing values using a safe map (no merge → no row multiplication)
    src_idx = src.set_index("dialog_id")
    for c in ensure_cols:
        # Only fill NaN/NA/empty strings
        s = assign_df[c]

        if s.dtype == object:
            mask = s.isna() | (s.astype(str).str.strip() == "")  # treat "" as missing
        else:
            mask = s.isna()

        if mask.any():
            mapped = assign_df.loc[mask, "dialog_id"].map(src_idx[c])
            assign_df.loc[mask, c] = mapped

    # Normalize types for is_gold/is_imc (optional but robust)
    for c in ["is_gold", "is_imc"]:
        if c in assign_df.columns:
            assign_df[c] = pd.to_numeric(assign_df[c], errors="coerce").fillna(0).astype(int)

    # Clamp remaining_slots to >= 0 (safety)
    assign_df["remaining_slots"] = pd.to_numeric(assign_df["remaining_slots"], errors="coerce").fillna(0)
    assign_df["remaining_slots"] = assign_df["remaining_slots"].clip(lower=0).astype(int)

    # Persist
    assign_df.to_csv(ASSIGN_CSV, index=False, encoding="utf-8")
    logger.info("[ASSIGN MIGRATION] ASSIGN_CSV migrated/validated and saved.")

    display(assign_df.head())

,dialog_id,recipe_title,recipe_type,condition,is_gold,is_imc,remaining_slots
0,D001,Moong Dal,savory,good,1,0,2
1,D002,Bourbon Wieners,sweet,good,0,0,2
2,D003,Cobb Salad Ham Roll-ups,savory,good,0,0,2
3,D004,Grain-Free Apple Cinnamon Dutch Babies,sweet,good,0,0,2
4,D005,Best Ever Muffins,sweet,good,1,0,2


In [ ]:
# Check remaining slots csv
import pandas as pd

assign_df = pd.read_csv(ASSIGN_CSV)

print("Shape:", assign_df.shape)
print("remaining_slots unique:", sorted(assign_df["remaining_slots"].unique())[:20])

# Check duplicates
dups = assign_df["dialog_id"].value_counts()
print("Dialog IDs with duplicates:", (dups > 1).sum())

if (dups > 1).any():
    print("\nExamples of duplicated dialog_ids:")
    print(dups[dups > 1].head(10))

# Show the first rows where remaining_slots != 2
print("\nRows with remaining_slots != 2:")
display(assign_df[assign_df["remaining_slots"] != 2].head(20))

Shape: (80, 7)
remaining_slots unique: [np.int64(2)]
Dialog IDs with duplicates: 0

Rows with remaining_slots != 2:


,dialog_id,recipe_title,recipe_type,condition,is_gold,is_imc,remaining_slots
